In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score

In [3]:
# 1. Load dữ liệu
df = pd.read_csv('../../data/processed/training_data_final.csv', parse_dates=['Datetime'], index_col='Datetime')

In [4]:
# ======================================================
# 2. FEATURE ENGINEERING (Tạo độ trễ - Lag)
# ======================================================
cols = ['Gold', 'Brent', 'Silver', 'Wheat', 'USD index']
target_col = 'Gold'

# Tạo Lag 1 đến Lag 3 (Dùng giá 3 phút trước để đoán giá hiện tại)
for col in cols:
    for lag in [1, 2, 3]:
        df[f'{col}_Lag{lag}'] = df[col].shift(lag)

# Xóa các dòng NaN
df.dropna(inplace=True)

# Xác định Input (X) và Output (y)
feature_cols = [c for c in df.columns if 'Lag' in c]
X = df[feature_cols]
y = df[target_col]

print(f"Kích thước dữ liệu sau xử lý: {df.shape}")

Kích thước dữ liệu sau xử lý: (177859, 20)


In [5]:
# ======================================================
# 3. CHIA TẬP TRAIN/TEST VÀ CHUẨN HÓA (SCALING)
# ======================================================
# Random Forest không bắt buộc phải scale X, nhưng ta scale y 
# để giữ đồng bộ code inverse_transform với các file khác.

split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# --- Scale X ---
scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# --- Scale y (Quan trọng cho bước Inverse sau này) ---
scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

# Ravel về mảng 1 chiều cho RF
y_train_scaled = y_train_scaled.ravel() 

print("✅ Đã chuẩn hóa dữ liệu.")

✅ Đã chuẩn hóa dữ liệu.


In [6]:
# ======================================================
# 4. HUẤN LUYỆN MÔ HÌNH RANDOM FOREST
# ======================================================
# n_estimators=100: Số lượng cây
# random_state=42: Để kết quả cố định mỗi lần chạy
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train_scaled)

print("✅ Huấn luyện xong!")

# (Optional) Xem mức độ quan trọng của các đặc trưng
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
print("\nTop 5 đặc trưng quan trọng nhất:")
for i in range(5):
    print(f"{feature_cols[indices[i]]}: {importances[indices[i]]:.4f}")

✅ Huấn luyện xong!

Top 5 đặc trưng quan trọng nhất:
Gold_Lag1: 0.8552
USD index_Lag1: 0.0396
Silver_Lag1: 0.0280
Gold_Lag2: 0.0213
Wheat_Lag1: 0.0204


In [7]:
# ======================================================
# 5. DỰ BÁO VÀ LƯU KẾT QUẢ
# ======================================================
# a. Dự báo
y_pred_scaled = model.predict(X_test_scaled)

# b. Inverse Transform (Đưa về giá USD thật)
pred_usd = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1))
actual_usd = scaler_y.inverse_transform(y_test_scaled.reshape(-1, 1))

# c. Tạo DataFrame kết quả
result_df = pd.DataFrame({
    'Datetime': y_test.index,
    'Actual': actual_usd.flatten(),
    'Predicted': pred_usd.flatten()
})

# d. Lưu file CSV
save_path = '../../results/predictions/RF.csv'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
result_df.to_csv(save_path, index=False)

print(f"\n✅ Đã lưu kết quả (USD) vào: {save_path}")
print(result_df.tail())


✅ Đã lưu kết quả (USD) vào: ../../results/predictions/RF.csv
                 Datetime   Actual    Predicted
35567 2025-12-18 23:57:00  4357.14  4209.284658
35568 2025-12-18 23:57:10  4358.14  4209.284658
35569 2025-12-18 23:57:20  4359.20  4209.284658
35570 2025-12-18 23:57:30  4359.63  4209.284658
35571 2025-12-18 23:57:40  4359.63  4209.284658
